# **Transformer Text Classification**

- https://docs.pytorch.org/docs/stable/generated/torch.nn.TransformerEncoder.html#torch.nn.TransformerEncoder

- https://docs.pytorch.org/docs/stable/generated/torch.nn.TransformerEncoderLayer.html#torch.nn.TransformerEncoderLayer

# **Model definition**

Recap (Sentiment with LSTM):


```
import torch
import torch.nn as nn

class Textclassifier(nn.Module):
    def __init__(self,vocabSize,dimension):
        super().__init__()
        self.dimension = dimension
        self.vocabSize = vocabSize

        self.lstm = nn.LSTM(self.dimension,self.dimension,batch_first=True)
        
        self.embedding = nn.Embedding(self.vocabSize,self.dimension)

        self.classification = nn.Linear(self.dimension,2)

    def forward(self,ins):
        embeds = self.embedding(ins)
        output, (h_n, c_n) = self.lstm(embeds)
        return self.classification(h_n[0])
```



In [1]:
import torch
import torch.nn as nn

class Textclassifier(nn.Module):
  def __init__(self,vocabSize,dimension,nrHeads,nrLayers):
    super().__init__()

    self.dimension = dimension
    self.vocabSize = vocabSize
    self.nrHeads = nrHeads
    self.nrLayers = nrLayers

    self.embedding = nn.Embedding(self.vocabSize,self.dimension)

    self.encoderLayer = nn.TransformerEncoderLayer(self.dimension,self.nrHeads)
    self.encoder = nn.TransformerEncoder(self.encoderLayer,self.nrLayers)

    self.classification = nn.Linear(self.dimension,2)

  def forward(self,ins):
    embeds = self.embedding(ins)
    transformerOut = self.encoder(embeds)
    return self.classification(transformerOut[0]) #this works because the vanilla pytorch transformer encoder has the sequence dimension first


In [2]:
testInput = torch.IntTensor([0,1,2,3,4,5])
classifier = Textclassifier(6,12,2,2)
output = classifier(testInput)

print (output)

tensor([ 0.0300, -0.4869], grad_fn=<ViewBackward0>)


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


**Text preparation**

Similar to the Transformer-based Bert model, we add a special [CLS] Token in the beginning of the input sequence, which will be used as input for the final classification.

In [3]:
corpus = ["this exercise is great","i am sad that this exercise will be over soon", "we love doing homework", "this homework was too hard"]
targets = torch.LongTensor([1,0,1,0])

def corpusToIndices(corpus):
    wordIndexDict = {}
    newCorpus = []
    index = 0
    for text in corpus:
        indexed = []
        clsPlusText = text.split()
        clsPlusText.insert(0,"[CLS]")
        for word in clsPlusText:
            if not word in wordIndexDict.keys():
                wordIndexDict[word] = index
                index += 1
            indexed.append(wordIndexDict[word])
        newCorpus.append(indexed)
    return newCorpus, wordIndexDict

newcorpus, wordIndexDict = corpusToIndices(corpus)

In [4]:
model = Textclassifier(20,12,2,2) #model init -> word vecs in 8 dimension
lossFct = nn.CrossEntropyLoss() #loss init
optimizer = torch.optim.AdamW(model.parameters(),lr=0.005) #optim init

for _ in range(15): #training loop for 15 epochs
  lossAbs = 0
  for text, target in zip(newcorpus,targets): #iterate over data and targets
    inputs = torch.IntTensor([text]) #transform inputs to tensor
    outs = model(inputs.squeeze_()) #feed inputs through model
    loss = lossFct(outs,target) #calculate loss
    loss.backward() #backward
    optimizer.step() #update
    lossAbs += loss
    optimizer.zero_grad()
  print (lossAbs/len(newcorpus))

posSentence = "[CLS] i love this exercise"
negSentence = "[CLS] this exercise is too hard"

posSentence = [wordIndexDict[x] for x in posSentence.split()]
print (posSentence)
negSentence = [wordIndexDict[x] for x in negSentence.split()]
print (negSentence)

print ("\nClassification Results:\nExpected 1 and 0")

print (model(torch.IntTensor(posSentence)))
print (model(torch.IntTensor(negSentence)))

tensor(1.2712, grad_fn=<DivBackward0>)
tensor(0.7787, grad_fn=<DivBackward0>)
tensor(0.7657, grad_fn=<DivBackward0>)
tensor(0.6806, grad_fn=<DivBackward0>)
tensor(0.7173, grad_fn=<DivBackward0>)
tensor(0.6611, grad_fn=<DivBackward0>)
tensor(0.7145, grad_fn=<DivBackward0>)
tensor(0.6822, grad_fn=<DivBackward0>)
tensor(0.7703, grad_fn=<DivBackward0>)
tensor(0.6432, grad_fn=<DivBackward0>)
tensor(0.6585, grad_fn=<DivBackward0>)
tensor(0.5557, grad_fn=<DivBackward0>)
tensor(0.4211, grad_fn=<DivBackward0>)
tensor(0.3236, grad_fn=<DivBackward0>)
tensor(0.2086, grad_fn=<DivBackward0>)
[0, 5, 14, 1, 2]
[0, 1, 2, 3, 18, 19]

Classification Results:
Expected 1 and 0
tensor([ 1.4282, -0.6134], grad_fn=<ViewBackward0>)
tensor([ 1.2146, -0.8842], grad_fn=<ViewBackward0>)


# **Using pre-trained models**

In [5]:
from transformers import BertTokenizer, BertModel
import torch

# Load pre-trained tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Example sentence
text = "Hello, how are you doing today?"

# Tokenize the input text
# add_special_tokens=True will add [CLS] and [SEP] tokens
# return_tensors='pt' will return PyTorch tensors
encoded_input = tokenizer(text, return_tensors='pt', add_special_tokens=True)

# Get model outputs
# The output will include last_hidden_state, pooler_output, and hidden_states (if output_hidden_states=True)
outputs = model(**encoded_input)

# The last_hidden_state contains the contextualized embeddings for all tokens
last_hidden_state = outputs.last_hidden_state

print("Original text:", text)
print("Tokenized input IDs:", encoded_input['input_ids'])
print("Tokenized input tokens:", tokenizer.convert_ids_to_tokens(encoded_input['input_ids'][0]))
print("Last hidden state:", last_hidden_state.shape)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Original text: Hello, how are you doing today?
Tokenized input IDs: tensor([[ 101, 7592, 1010, 2129, 2024, 2017, 2725, 2651, 1029,  102]])
Tokenized input tokens: ['[CLS]', 'hello', ',', 'how', 'are', 'you', 'doing', 'today', '?', '[SEP]']
Last hidden state: torch.Size([1, 10, 768])


# **Fine-tuning a pre-trained model**

Defining the model with pre-trained building blocks:

In [6]:
import torch
import torch.nn as nn

class Textclassifier2(nn.Module):
  def __init__(self):
    super().__init__()

    self.bert = BertModel.from_pretrained('bert-base-uncased')
    self.classification = nn.Linear(768,2)

  def forward(self,ins):
    outputs = self.bert(**ins)
    last_hidden_state = outputs.last_hidden_state
    return self.classification(last_hidden_state[:,0,:]) #note that the batch is in the first dimension, and the second dimension is the sequence dimension

In [7]:
text = "This exercise is too hard"
encoded_input = tokenizer(text, return_tensors='pt', add_special_tokens=True)
print (encoded_input)
classifier2 = Textclassifier2()
print (classifier2(encoded_input))

{'input_ids': tensor([[ 101, 2023, 6912, 2003, 2205, 2524,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]])}
tensor([[-0.4329,  0.1058]], grad_fn=<AddmmBackward0>)


**Train Loop**

In [8]:
model = Textclassifier2() #model init -> word vecs in 8 dimension
lossFct = nn.CrossEntropyLoss() #loss init
optimizer = torch.optim.AdamW(model.parameters(),lr=0.0001) #optim init

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased') #bert model tokenizer

corpus = ["this exercise is great","i am sad that this exercise will be over soon", "we love doing homework", "this homework was too hard"]
targets = torch.LongTensor([1,0,1,0])

for _ in range(10): #training loop for x epochs
  lossAbs = 0
  for text, target in zip(corpus,targets): #iterate over data and targets
    inputs = tokenizer(text, return_tensors='pt', add_special_tokens=True)
    outs = model(inputs) #feed inputs through model
    loss = lossFct(outs.squeeze(),target) #calculate loss
    loss.backward() #backward
    optimizer.step() #update
    lossAbs += loss
    optimizer.zero_grad()
  print (lossAbs/len(newcorpus))

posSentence = "i love this exercise"
negSentence = "this exercise is too hard"


print ("\nClassification Results:\nExpected 1 and 0")

print (model(tokenizer(posSentence, return_tensors='pt', add_special_tokens=True)))
print (model(tokenizer(negSentence, return_tensors='pt', add_special_tokens=True)))

tensor(0.8271, grad_fn=<DivBackward0>)
tensor(0.5738, grad_fn=<DivBackward0>)
tensor(0.2983, grad_fn=<DivBackward0>)
tensor(0.0200, grad_fn=<DivBackward0>)
tensor(0.0038, grad_fn=<DivBackward0>)
tensor(0.0011, grad_fn=<DivBackward0>)
tensor(0.0006, grad_fn=<DivBackward0>)
tensor(0.0004, grad_fn=<DivBackward0>)
tensor(0.0003, grad_fn=<DivBackward0>)
tensor(0.0002, grad_fn=<DivBackward0>)

Classification Results:
Expected 1 and 0
tensor([[-4.2589,  3.3513]], grad_fn=<AddmmBackward0>)
tensor([[ 3.7918, -3.8347]], grad_fn=<AddmmBackward0>)


Training the complete model including the Bert weights takes a considerable amount of time. Sometimes it is useful to exclude some weights from training

In [9]:
import torch
import torch.nn as nn

class Textclassifier3(nn.Module):
  def __init__(self):
    super().__init__()

    with torch.no_grad():
      self.bert = BertModel.from_pretrained('bert-base-uncased')
    self.classification = nn.Linear(768,2)

  def forward(self,ins):
    outputs = self.bert(**ins)
    last_hidden_state = outputs.last_hidden_state
    return self.classification(last_hidden_state[:,0,:]) #note that the batch is in the first dimension, and the second dimension is the sequence dimension

In [10]:
model = Textclassifier3() #model init -> word vecs in 8 dimension
lossFct = nn.CrossEntropyLoss() #loss init
optimizer = torch.optim.AdamW(model.parameters(),lr=0.0001) #optim init

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased') #bert model tokenizer

corpus = ["this exercise is great","i am sad that this exercise will be over soon", "we love doing homework", "this homework was too hard"]
targets = torch.LongTensor([1,0,1,0])

for _ in range(10): #training loop for x epochs
  lossAbs = 0
  for text, target in zip(corpus,targets): #iterate over data and targets
    inputs = tokenizer(text, return_tensors='pt', add_special_tokens=True)
    outs = model(inputs) #feed inputs through model
    loss = lossFct(outs.squeeze(),target) #calculate loss
    loss.backward() #backward
    optimizer.step() #update
    lossAbs += loss
    optimizer.zero_grad()
  print (lossAbs/len(newcorpus))

posSentence = "i love this exercise"
negSentence = "this exercise is too hard"


print ("\nClassification Results:\nExpected 1 and 0")

print (model(tokenizer(posSentence, return_tensors='pt', add_special_tokens=True)))
print (model(tokenizer(negSentence, return_tensors='pt', add_special_tokens=True)))

tensor(1.1403, grad_fn=<DivBackward0>)
tensor(0.5482, grad_fn=<DivBackward0>)
tensor(0.1159, grad_fn=<DivBackward0>)
tensor(0.0172, grad_fn=<DivBackward0>)
tensor(0.0050, grad_fn=<DivBackward0>)
tensor(0.0025, grad_fn=<DivBackward0>)
tensor(0.0015, grad_fn=<DivBackward0>)
tensor(0.0010, grad_fn=<DivBackward0>)
tensor(0.0008, grad_fn=<DivBackward0>)
tensor(0.0006, grad_fn=<DivBackward0>)

Classification Results:
Expected 1 and 0
tensor([[-3.6071,  3.9758]], grad_fn=<AddmmBackward0>)
tensor([[ 2.8485, -2.7096]], grad_fn=<AddmmBackward0>)


Alternatively, gradients for individual layers can be deactivated:

In [11]:
bert = BertModel.from_pretrained('bert-base-uncased')

print (bert)

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [12]:
for param in bert.embeddings.parameters(): #deactivating backprop for the embedding layer
  param.requires_grad=False